## Avaliando o Modelo RAG com o framework Ragas

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Importante! Versão do RAGAS deve ser 0.1.21
!pip install ragas==0.1.21 pandas openpyxl datasets langchain openai

In [ ]:
import pandas as pd
import ast
import os

from openai import OpenAI
from datasets import Dataset
from langchain.chat_models import ChatOpenAI

In [ ]:
os.environ["OPENAI_API_KEY"] = "sua-chave-openai-aqui"
openai_client = OpenAI(api_key="OPENAI_API_KEY")

In [ ]:
from ragas import evaluate
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory

from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

In [ ]:
# --------------------------------------------------
# 1. Função para converter a coluna 'contexts'
# --------------------------------------------------
def parse_contexts(context_str):
    """
    Converte string do tipo:
    "'ctx1','ctx2','ctx3'"
    para lista Python:
    ["ctx1", "ctx2", "ctx3"]
    """
    try:
        # Garante formato de lista para ast.literal_eval
        formatted = f"[{context_str}]"
        return ast.literal_eval(formatted)
    except Exception as e:
        print(f"Erro ao converter contextos: {context_str}")
        return []


In [ ]:
# --------------------------------------------------
# 2. Lendo o arquivo Excel
# --------------------------------------------------
df = pd.read_excel("/MBA Esalq/TCC/EXPERIMENTOS/experimento-10.xlsx")

In [ ]:
df.head()

In [ ]:
# Convertendo os contexts
df["contexts"] = df["contexts"].apply(parse_contexts)

In [ ]:
df.head()

In [ ]:
df = df.rename(columns={
    "reference": "ground_truth"
})

In [ ]:
# --------------------------------------------------
# 3. Convertendo para o formato esperado pelo RAGAS
# --------------------------------------------------
dataset = Dataset.from_pandas(df)

In [ ]:
# --------------------------------------------------
# 4. Criando a LLM
# --------------------------------------------------
llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",  # mais barato
    temperature=0
)

In [ ]:
# --------------------------------------------------
# 5. Executando a avaliação
# --------------------------------------------------
result = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=llm
)

In [ ]:
# --------------------------------------------------
# 6. Exibindo os resultados
# --------------------------------------------------
print(result)

In [ ]:
# Se quiser transformar em DataFrame
result_df = result.to_pandas()
print(result_df.head())

In [ ]:
# Salvando em Excel (opcional)
result_df.to_excel("/MBA Esalq/TCC/EXPERIMENTOS/resultado-ragas-experimento-1.xlsx", index=False)